## **1. Import necessary libraries**

In [ ]:
%pip install beautifulsoup4

import os
import requests
import time
import random
from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

root_dir = './vn_news_corpus'
os.makedirs(root_dir, exist_ok=True)
n_pages = 10
article_id = 0

## **2. Crawl**

In [ ]:
def fetch_html_with_fallback(url, driver, retries=3, page_timeout=30, req_timeout=20):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    for attempt in range(retries):
        try:
            driver.set_page_load_timeout(page_timeout)
            driver.get(url)
            WebDriverWait(driver, page_timeout).until(lambda d: d.execute_script("return document.readyState") == "complete")
            return driver.page_source
        except Exception:
            time.sleep(1 + random.random()*2)
    try:
        r = requests.get(url, headers=headers, timeout=req_timeout)
        r.raise_for_status()
        return r.text
    except Exception:
        return None

for page_idx in tqdm(range(n_pages)):
    main_url = f'https://vietnamnet.vn/thoi-su-page{page_idx}'
    page_html = fetch_html_with_fallback(main_url, driver)
    if not page_html:
        continue
    page_soup = BeautifulSoup(page_html, 'html.parser')
    news_tags = page_soup.select('div.topStory-15nd div div a')
    news_page_urls = [a.get('href') for a in news_tags if a.get('href')]

    for news_page_url in news_page_urls:
        html = fetch_html_with_fallback(news_page_url, driver)
        if not html:
            continue
        soup = BeautifulSoup(html, 'lxml')
        main_content = soup.select_one('div.content-detail')
        if not main_content:
            continue
        if main_content.select_one('div.video-detail'):
            continue

        title_tag = main_content.find('h1')
        title = title_tag.get_text(strip=True) if title_tag else ''
        abstract_tag = main_content.find('h2')
        abstract = abstract_tag.get_text(strip=True) if abstract_tag else ''
        author_tag = main_content.select_one('span.name')
        author = author_tag.get_text(strip=True) if author_tag else ''
        paragraphs_tags = main_content.select('div.maincontent.main-content p')
        if not paragraphs_tags:
            paragraphs_tags = main_content.find_all('p')
        paragraphs_lst = [p.get_text(strip=True) for p in paragraphs_tags]
        paragraphs = ' '.join([p for p in paragraphs_lst if p])

        parts = [part for part in [title, abstract, author, paragraphs] if part]
        final_content = '\n\n'.join(parts)

        article_filename = f'article_{article_id:05d}.txt'
        article_savepath = os.path.join(root_dir, article_filename)
        article_id += 1
        with open(article_savepath, 'w', encoding='utf-8') as f:
            f.write(final_content)

        time.sleep(0.5 + random.random()*1.0)

100%|██████████| 10/10 [19:13<00:00, 115.33s/it]


In [ ]:
!zip -r vn_news_corpus.zip vn_news_corpus

  adding: vn_news_corpus/ (stored 0%)
  adding: vn_news_corpus/article_00070.txt (deflated 50%)
  adding: vn_news_corpus/article_00030.txt (deflated 56%)
  adding: vn_news_corpus/article_00006.txt (deflated 55%)
  adding: vn_news_corpus/article_00091.txt (deflated 48%)
  adding: vn_news_corpus/article_00060.txt (deflated 70%)
  adding: vn_news_corpus/article_00037.txt (deflated 58%)
  adding: vn_news_corpus/article_00073.txt (deflated 55%)
  adding: vn_news_corpus/article_00114.txt (deflated 66%)
  adding: vn_news_corpus/article_00110.txt (deflated 63%)
  adding: vn_news_corpus/article_00111.txt (deflated 58%)
  adding: vn_news_corpus/article_00034.txt (deflated 60%)
  adding: vn_news_corpus/article_00090.txt (deflated 53%)
  adding: vn_news_corpus/article_00011.txt (deflated 58%)
  adding: vn_news_corpus/article_00069.txt (deflated 66%)
  adding: vn_news_corpus/article_00099.txt (deflated 46%)
  adding: vn_news_corpus/article_00039.txt (deflated 70%)
  adding: vn_news_corpus/article_0

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!cp '/content/vn_news_corpus.zip' '/content/gdrive/MyDrive/Coordinate/aio_2023_ta/module1/data_handling_project/dataset'